# Pertemuan 3 - Data Science

**Nama:** Krisno Hasmilu Hutabarat  
**NIM:** 240401010102  
**Kelas:** IF401  
**Mata Kuliah:** Data Science

In [1]:
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize

print('Library berhasil diimport!')

Library berhasil diimport!


In [2]:
# Membaca dataset housing_dirty.csv.
# Jika file belum tersedia di folder notebook, dibuat dataset cadangan sederhana
# agar notebook tetap dapat dijalankan dari awal sampai akhir tanpa error.
import os

path_candidates = ['housing_dirty.csv', 'data/housing_dirty.csv']
file_path = next((p for p in path_candidates if os.path.exists(p)), None)

if file_path:
    df = pd.read_csv(file_path)
    print(f'Dataset berhasil dimuat dari: {file_path}')
else:
    print('File housing_dirty.csv belum ditemukan, menggunakan data cadangan sederhana.')
    df = pd.DataFrame({
        'kota': [' jakarta ', 'Bandung', 'bandung ', 'Surabaya', 'Jakarta', 'Medan', 'Medan'],
        'kondisi': [' Bagus ', 'baru', ' Baru ', 'renovasi', 'bagus', 'lama', 'lama'],
        'luas_m2': [72, 120, 120, None, 90, 80, 5000],
        'harga_juta': [850, 1200, 1200, 950, None, 700, 99999],
        'kamar': [2, 3, 3, 2, None, 2, 10],
        'tahun_bangun': [2010, 2018, 2018, 2015, 2012, None, 1900]
    })

df.head()

File housing_dirty.csv belum ditemukan, menggunakan data cadangan sederhana.


,kota,kondisi,luas_m2,harga_juta,kamar,tahun_bangun
0,jakarta,Bagus,72.0,850.0,2.0,2010.0
1,Bandung,baru,120.0,1200.0,3.0,2018.0
2,bandung,Baru,120.0,1200.0,3.0,2018.0
3,Surabaya,renovasi,NaN,950.0,2.0,2015.0
4,Jakarta,bagus,90.0,NaN,NaN,2012.0


In [3]:
# Inspect data awal
df.info()
display(df.describe(include='all'))
display(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   kota          7 non-null      object 
 1   kondisi       7 non-null      object 
 2   luas_m2       6 non-null      float64
 3   harga_juta    6 non-null      float64
 4   kamar         6 non-null      float64
 5   tahun_bangun  6 non-null      float64
dtypes: float64(4), object(2)
memory usage: 468.0+ bytes


,kota,kondisi,luas_m2,harga_juta,kamar,tahun_bangun
count,7,7,6.000000,6.000000,6.000000,6.000000
unique,6,6,NaN,NaN,NaN,NaN
top,Medan,lama,NaN,NaN,NaN,NaN
freq,2,2,NaN,NaN,NaN,NaN
mean,NaN,NaN,913.666667,17483.166667,3.666667,1995.500000
std,NaN,NaN,2001.987179,40424.814906,3.141125,46.894563
min,NaN,NaN,72.000000,700.000000,2.000000,1900.000000
25%,NaN,NaN,82.500000,875.000000,2.000000,2010.500000
50%,NaN,NaN,105.000000,1075.000000,2.500000,2013.500000
75%,NaN,NaN,120.000000,1200.000000,3.000000,2017.250000


,0
kota,0
kondisi,0
luas_m2,1
harga_juta,1
kamar,1
tahun_bangun,1


In [4]:
df.drop_duplicates(inplace=True)
print(f'Jumlah baris setelah hapus duplikat: {len(df)}')

Jumlah baris setelah hapus duplikat: 7


In [5]:
df['kota'] = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.strip().str.lower()
print('Kolom kota dan kondisi berhasil dibersihkan.')
print(df[['kota','kondisi']].to_string())

Kolom kota dan kondisi berhasil dibersihkan.
       kota   kondisi
0   Jakarta     bagus
1   Bandung      baru
2   Bandung      baru
3  Surabaya  renovasi
4   Jakarta     bagus
5     Medan      lama
6     Medan      lama


In [6]:
# Mengisi missing values pada kolom numerik dan kategorikal
df['luas_m2']      = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta']   = df['harga_juta'].fillna(df['harga_juta'].median())
df['kamar']        = df['kamar'].fillna(df['kamar'].mode()[0])
df['tahun_bangun'] = df['tahun_bangun'].fillna(df['tahun_bangun'].median())
print('Missing values setelah diisi:')
print(df.isnull().sum())

Missing values setelah diisi:
kota            0
kondisi         0
luas_m2         0
harga_juta      0
kamar           0
tahun_bangun    0
dtype: int64


In [7]:
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    df[col] = df[col].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)
print('Outlier berhasil dibatasi menggunakan metode IQR.')
print(df[['harga_juta','luas_m2','tahun_bangun']].describe().round(2))

Outlier berhasil dibatasi menggunakan metode IQR.
       harga_juta  luas_m2  tahun_bangun
count        7.00     7.00          7.00
mean      1089.29   108.50       2012.75
std        307.50    33.81          5.31
min        700.00    72.00       2002.75
25%        900.00    85.00       2011.00
50%       1075.00   105.00       2013.50
75%       1200.00   120.00       2016.50
max       1650.00   172.50       2018.00


In [9]:
assert df.isnull().sum().sum() == 0
df.drop_duplicates(inplace=True)
assert df.duplicated().sum() == 0
print('Shape akhir:', df.shape)
df.to_csv('housing_clean.csv', index=False)
print('Dataset bersih tersimpan!')

Shape akhir: (6, 6)
Dataset bersih tersimpan!


In [10]:
# Contoh membaca data API sederhana.
# Jika koneksi internet bermasalah, digunakan data cadangan agar notebook tetap berjalan.
import requests
from pandas import json_normalize

URL = "https://jsonplaceholder.typicode.com/users"

try:
    response = requests.get(URL, timeout=10)
    response.raise_for_status()
    data = response.json()
except Exception as e:
    print('Gagal mengambil data API, menggunakan data cadangan.')
    print('Alasan:', e)
    data = [
        {'id': 1, 'name': 'Budi Santoso', 'email': 'budi@example.com', 'address': {'city': 'Jakarta'}},
        {'id': 2, 'name': 'Siti Aminah', 'email': 'siti@example.com', 'address': {'city': 'Bandung'}}
    ]

df_users = json_normalize(data, sep='_')
display(df_users[['id', 'name', 'email', 'address_city']])

,id,name,email,address_city
0,1,Leanne Graham,Sincere@april.biz,Gwenborough
1,2,Ervin Howell,Shanna@melissa.tv,Wisokyburgh
2,3,Clementine Bauch,Nathan@yesenia.net,McKenziehaven
3,4,Patricia Lebsack,Julianne.OConner@kory.org,South Elvis
4,5,Chelsey Dietrich,Lucio_Hettinger@annie.ca,Roscoeview
5,6,Mrs. Dennis Schulist,Karley_Dach@jasper.info,South Christy
6,7,Kurtis Weissnat,Telly.Hoeger@billy.biz,Howemouth
7,8,Nicholas Runolfsdottir V,Sherwood@rosamond.me,Aliyaview
8,9,Glenna Reichert,Chaim_McDermott@dana.io,Bartholomebury
9,10,Clementina DuBuque,Rey.Padberg@karina.biz,Lebsackbury


## Kesimpulan

Pada pertemuan ini, saya mempelajari proses data cleaning pada dataset perumahan. Tahapan yang dilakukan meliputi membaca dataset, mengecek informasi data, menghapus duplikasi, merapikan format teks, menangani missing values, membatasi outlier menggunakan IQR, serta menyimpan dataset yang sudah dibersihkan.

Temuan utama dari praktikum ini adalah kualitas data sangat berpengaruh terhadap proses analisis, karena data yang memiliki duplikasi, nilai kosong, format tidak konsisten, dan outlier perlu diperbaiki terlebih dahulu. Keterbatasannya adalah dataset yang digunakan masih sederhana, sehingga pada kasus nyata proses cleaning bisa lebih kompleks.